# Bài 2 – Trang 32: Tính F-measure

**6451071011 - Lê Nguyễn Anh Dự**

Tính F1 từ precision và recall cho các bài đã thực hiện: Adult Income (Bài 2 trang 15), Red Wine Quality (Bài 4 trang 26), Heart Disease (Bài 5 trang 26), Mushroom (Bài 6 trang 27) và Digits (Bài 7 trang 27). Chạy các ô theo thứ tự trong Google Colab.

**F1 = 2 ? precision ? recall / (precision + recall)**

Với bài nhị phân, F1 được tính cho lớp cần quan tâm (`>50K`, bệnh tim `1`, nấm độc `p`). Với bài nhiều lớp, dùng **macro average**: trung bình F1 của từng lớp. Nếu precision + recall = 0, quy ước F1 = 0.


In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import load_digits
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

STUDENT = '6451071011 - Lê Nguyễn Anh Dự'
FILE_IDS = {
    'adult.data.csv': '1BYKP73pyzphA5YHR1fVvn2A0kBpr09TB',
    'adult.test.csv': '1Vh7e1IBL0SRxjT3gU_v_rnIcAT18griD',
    'winequality-red.csv': '1rzLiUi64RLmMV8ApudCt6aJyv_OkhADH',
    'heart.csv': '10N9jGzMn-MagDWXORdP-qIpwC3omZJHb',
    'mushrooms.csv': '11OXFpnwUwK_Ka9BFmboMoT9RhTRe3CFW'
}

def heading(number, title):
    print(STUDENT)
    print(f'Câu {number}. {title}')

def ensure_csv(name):
    if not os.path.exists(name):
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
            import gdown
        gdown.download(id=FILE_IDS[name], output=name, quiet=False)
    if not os.path.exists(name):
        raise FileNotFoundError(f'Không tải được {name}; hãy tải tệp lên Colab.')
    return name

def f_from_pr(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

rows = []
def score_model(exercise, name, model, X_train, X_test, y_train, y_test,
                pos_label=None):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    if pos_label is None:
        p = precision_score(y_test, pred, average='macro', zero_division=0)
        r = recall_score(y_test, pred, average='macro', zero_division=0)
        f = f1_score(y_test, pred, average='macro', zero_division=0)
        kind = 'macro'
        # Macro F1 = trung bình F1 từng lớp; thường khác harmonic mean của macro P/R.
        direct = f_from_pr(p, r)
    else:
        p = precision_score(y_test, pred, pos_label=pos_label, zero_division=0)
        r = recall_score(y_test, pred, pos_label=pos_label, zero_division=0)
        f = f1_score(y_test, pred, pos_label=pos_label, zero_division=0)
        kind = f'lớp {pos_label}'
        direct = f_from_pr(p, r)
        assert np.isclose(f, direct)
    rows.append({'Bài': exercise, 'Mô hình': name, 'Cách tính': kind,
                 'Precision': p, 'Recall': r,
                 '2PR/(P+R)': direct, 'F1': f,
                 'Accuracy': accuracy_score(y_test, pred)})

def show_exercise(exercise):
    table = pd.DataFrame([r for r in rows if r['Bài'] == exercise])
    display(table.style.format({'Precision':'{:.4f}', 'Recall':'{:.4f}',
                                '2PR/(P+R)':'{:.4f}', 'F1':'{:.4f}',
                                'Accuracy':'{:.4f}'}))


## Câu 1. Adult Income – Bài 2 trang 15

Giữ đúng ranh giới train/test gốc sau khi xóa dòng có `?`; loại `fnlwgt`, khảo sát tương quan và mã hóa one-hot trên hai phần gộp như bài trước.


In [ ]:
heading(1, 'F-measure của Adult Income')
train = pd.read_csv(ensure_csv('adult.data.csv'), skipinitialspace=True)
test = pd.read_csv(ensure_csv('adult.test.csv'), skipinitialspace=True)
train.columns = train.columns.str.strip().str.lower()
test.columns = test.columns.str.strip().str.lower()
def clean_adult(df):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    return df.replace(r'^\?$', np.nan, regex=True).dropna().reset_index(drop=True)
train = clean_adult(train).drop(columns='fnlwgt')
test = clean_adult(test)
assert list(train.columns) == list(test.columns)
cut = len(train)
adult = pd.concat([train, test], ignore_index=True)
Xa = adult.drop(columns='income')
ya = adult['income'].str.rstrip('.')
numeric = Xa.select_dtypes(include=np.number)
upper = numeric.corr().abs().where(np.triu(np.ones((numeric.shape[1], numeric.shape[1])), 1).astype(bool))
Xa = Xa.drop(columns=[c for c in upper.columns if (upper[c] >= 0.90).any()])
Xa = pd.get_dummies(Xa, columns=Xa.select_dtypes(include='object').columns, dtype=np.uint8)
X_train, X_test = Xa.iloc[:cut], Xa.iloc[cut:]
y_train, y_test = ya.iloc[:cut], ya.iloc[cut:]
for name, model in [('ID3 (entropy)', DecisionTreeClassifier(criterion='entropy', random_state=42)),
                    ('CART (gini)', DecisionTreeClassifier(criterion='gini', random_state=42)),
                    ('GaussianNB', GaussianNB())]:
    score_model('Adult Income', name, model, X_train, X_test, y_train, y_test, '>50K')
show_exercise('Adult Income')


## Câu 2. Red Wine Quality – Bài 4 trang 26

In [ ]:
heading(2, 'F-measure của Red Wine Quality')
wine = pd.read_csv(ensure_csv('winequality-red.csv'), sep=None, engine='python')
wine.columns = wine.columns.str.strip()
Xw = wine.drop(columns='quality')
yw = wine['quality']
numeric = Xw.select_dtypes(include=np.number)
upper = numeric.corr().abs().where(np.triu(np.ones((numeric.shape[1], numeric.shape[1])), 1).astype(bool))
Xw = Xw.drop(columns=[c for c in upper.columns if (upper[c] >= 0.90).any()])
Xw = pd.get_dummies(Xw, columns=Xw.select_dtypes(include='object').columns, dtype=np.uint8)
X_train, X_test, y_train, y_test = train_test_split(Xw, yw, test_size=0.3, random_state=42)
for name, model in [('ID3 (entropy)', DecisionTreeClassifier(criterion='entropy', random_state=42)),
                    ('CART (gini)', DecisionTreeClassifier(criterion='gini', random_state=42)),
                    ('GaussianNB', GaussianNB())]:
    score_model('Red Wine', name, model, X_train, X_test, y_train, y_test)
show_exercise('Red Wine')


## Câu 3. Heart Disease – Bài 5 trang 26

In [ ]:
heading(3, 'F-measure của Heart Disease')
heart = pd.read_csv(ensure_csv('heart.csv'), encoding='utf-8-sig', skipinitialspace=True)
heart.columns = ['age','sex','chest_pain_type','resting_blood_pressure','cholesterol',
                 'fasting_blood_sugar','rest_ecg','max_heart_rate_achieved',
                 'exercise_induced_angina','st_depression','st_slope',
                 'num_major_vessels','thalassemia','target']
heart = heart.replace(r'^\s*\?\s*$', np.nan, regex=True)
for col in heart.columns:
    heart[col] = pd.to_numeric(heart[col], errors='coerce')
heart = heart.dropna().reset_index(drop=True)
heart['target'] = heart['target'].astype(int)
Xh = heart.drop(columns='target')
yh = heart['target']
upper = Xh.corr().abs().where(np.triu(np.ones((Xh.shape[1], Xh.shape[1])), 1).astype(bool))
Xh = Xh.drop(columns=[c for c in upper.columns if (upper[c] >= 0.90).any()])
Xh = pd.get_dummies(Xh, columns=Xh.select_dtypes(include='object').columns, dtype=np.uint8)
X_train, X_test, y_train, y_test = train_test_split(Xh, yh, test_size=0.3, random_state=42)
for name, model in [('ID3 (entropy)', DecisionTreeClassifier(criterion='entropy', random_state=42)),
                    ('CART (gini)', DecisionTreeClassifier(criterion='gini', random_state=42)),
                    ('GaussianNB', GaussianNB())]:
    score_model('Heart Disease', name, model, X_train, X_test, y_train, y_test, 1)
show_exercise('Heart Disease')


## Câu 4. Mushroom – Bài 6 trang 27

In [ ]:
heading(4, 'F-measure của Mushroom')
mushroom = pd.read_csv(ensure_csv('mushrooms.csv'), skipinitialspace=True)
Xm = mushroom.drop(columns='class')
ym = mushroom['class']
Xm = pd.get_dummies(Xm, columns=Xm.select_dtypes(include='object').columns, dtype=np.uint8)
X_train, X_test, y_train, y_test = train_test_split(Xm, ym, test_size=0.3, random_state=42)
for name, model in [('ID3 (entropy)', DecisionTreeClassifier(criterion='entropy', random_state=42)),
                    ('CART (gini)', DecisionTreeClassifier(criterion='gini', random_state=42)),
                    ('GaussianNB', GaussianNB())]:
    score_model('Mushroom', name, model, X_train, X_test, y_train, y_test, 'p')
show_exercise('Mushroom')


## Câu 5. Digits – Bài 7 trang 27

Dùng đúng phép chia 80/20 không xáo trộn và cây entropy của notebook trước. Vì có 10 lớp, `F1 macro` là trung bình F1 riêng từng chữ số.


In [ ]:
heading(5, 'F-measure của Digits')
digits = load_digits()
Xd = digits.images.reshape(len(digits.images), -1)
yd = digits.target
X_train, X_test, y_train, y_test = train_test_split(Xd, yd, test_size=0.2, shuffle=False)
score_model('Digits', 'ID3 (entropy)',
            DecisionTreeClassifier(criterion='entropy', random_state=33),
            X_train, X_test, y_train, y_test)
show_exercise('Digits')


## Câu 6. Tổng hợp và nhận xét

Không so sánh trực tiếp F1 giữa các bộ dữ liệu khác nhau để chọn một mô hình chung: mỗi bài có độ khó và phân bố lớp riêng. So sánh các mô hình **trong cùng một bài**.


In [ ]:
heading(6, 'Tổng hợp F-measure các bài')
summary = pd.DataFrame(rows)
display(summary.style.format({'Precision':'{:.4f}', 'Recall':'{:.4f}',
                              '2PR/(P+R)':'{:.4f}', 'F1':'{:.4f}',
                              'Accuracy':'{:.4f}'}))
for exercise, group in summary.groupby('Bài', sort=False):
    best = group.loc[group['F1'].idxmax()]
    print(f'{exercise}: F1 cao nhất là {best["Mô hình"]} ({best["F1"]:.4f}).')
print('Lưu ý: Với bài nhiều lớp, cột 2PR/(P+R) dùng macro precision/recall để minh họa công thức; F1 macro là trung bình F1 của từng lớp nên có thể khác giá trị này.')
